In [1]:
import pandas as pd
import re
from pathlib import Path

In [2]:
DATA_DIR = Path(r"C:\Users\pietr\Desktop\github\2024-2025\crisis\code\mashup\sub_mash")
files = sorted(DATA_DIR.glob("chunk_*.csv"))
print(f"Found {len(files)} files:")
for f in files:
    print(" ", f.name)

Found 5 files:
  chunk_1.csv
  chunk_2.csv
  chunk_3.csv
  chunk_4.csv
  chunk_5.csv


In [3]:
iris_map = {
    "7.01 Carta tematica e geografica": [
        r"7\.01", r"carta\s+(tematica|geografica)"
    ],
    "7.02 Carta geologica": [r"7\.02", r"carta\s+geologica"],
    "7.03 Prodotto dell'ingegneria civile e dell'architettura": [
        r"7\.03", r"prodotto.*ingegneria", r"model", r"physical\s*object"
    ],
    "7.04 Software": [
        r"7\.04", r"\bsoftware\b", r"workflow",
        r"computational notebook", r"software\s*documentation"
    ],
    "7.05 Banche dati": [
        r"7\.05", r"banche\s+dati", r"dataset",
        r"annotation collection", r"database"
    ],
    "7.06 Composizione musicale": [
        r"7\.06", r"composizione\s+musicale", r"audio"
    ],
    "7.07 Disegno": [
        r"7\.07", r"disegno", r"(figure|image|diagram)"
    ],
    "7.08 Design": [r"7\.08", r"\bdesign\b"],
    "7.09 Performance": [r"7\.09", r"\bperformance\b", r"event"],
    "7.10 Manufatto": [
        r"7\.10", r"manufatto", r"(physical object|model|prototype)"
    ],
    "7.11 Prototipo d'arte e relativi progetti": [r"7\.11", r"prototipo"],
    "7.12 Mostra o Esposizione": [
        r"7\.12", r"(mostra|esposizione|exhibition|event)"
    ],
    "7.13 Rapporto tecnico": [
        r"7\.13", r"rapporto\s+tecnico",
        r"(technical note|project deliverable|report)"
    ],
    "7.14 Audiovisivi": [
        r"7\.14", r"(video|audiovisivi|video/audio|poster\+video)"
    ],
    "7.15 Test psicologici": [
        r"7\.15", r"(test psicologici|psychological test|psychometrics)"
    ],
}

In [4]:
def classify(row):
    txt = f"{row['type']} {row['resource_type']}".lower()
    for iris, patterns in iris_map.items():
        if any(re.search(pat, txt) for pat in patterns):
            return iris
    return pd.NA

In [5]:
chunksize = 100_000
matched_chunks   = []
discarded_chunks = []

for file in files:
    for chunk in pd.read_csv(file, chunksize=chunksize, low_memory=False):
        chunk['src_repo'] = chunk['src_repo'].str.lower()
        chunk['type']     = chunk['type'].str.lower()
        chunk["iris_cat"] = chunk.apply(classify, axis=1)

        matched_chunks.append(chunk.dropna(subset=["iris_cat"]))
        discarded_chunks.append(chunk[chunk["iris_cat"].isna()])

iris_df      = pd.concat(matched_chunks,   ignore_index=True)
discarded_df = pd.concat(discarded_chunks, ignore_index=True)

In [6]:
total       = len(iris_df) + len(discarded_df)
n_matched   = len(iris_df)
n_discarded = len(discarded_df)

print(f"Total rows processed : {total:,}")
print(f"Matched (IRIS)       : {n_matched:,}  ({n_matched/total*100:.1f}%)")
print(f"Discarded (no match) : {n_discarded:,}  ({n_discarded/total*100:.1f}%)")

Total rows processed : 419,824
Matched (IRIS)       : 8,697  (2.1%)
Discarded (no match) : 411,127  (97.9%)


In [7]:
discarded_breakdown = (
    discarded_df
    .groupby(['type', 'resource_type'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)

print(f"Distinct (type, resource_type) combinations in discarded rows: {len(discarded_breakdown)}")
discarded_breakdown

Distinct (type, resource_type) combinations in discarded rows: 74


,type,resource_type,count
0,1.01 articolo in rivista,NaN,213179
1,2.01 capitolo / saggio in libro,NaN,55454
2,4.01 contributo in atti di convegno,NaN,43967
3,4.02 riassunto (abstract),NaN,21410
4,3.01 monografia / trattato scientifico in form...,NaN,10072
...,...,...,...
69,monograph,catalogo,2
70,publication,standard,2
71,monograph,trattato,1
72,book_section,atti_convegno,1


In [8]:
coverage = (
    iris_df
    .groupby(['src_repo', 'iris_cat'])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)

coverage

iris_cat,7.01 Carta tematica e geografica,7.02 Carta geologica,7.03 Prodotto dell'ingegneria civile e dell'architettura,7.04 Software,7.05 Banche dati,7.06 Composizione musicale,7.07 Disegno,7.08 Design,7.09 Performance,7.10 Manufatto,7.11 Prototipo d'arte e relativi progetti,7.12 Mostra o Esposizione,7.13 Rapporto tecnico,7.15 Test psicologici
src_repo,,,,,,,,,,,,,,
amsacta,0,0,0,8,375,0,0,0,0,0,0,0,69,0
iris,120,73,225,589,258,148,16,10,185,156,9,1996,837,87
software heritage,0,0,0,1014,0,0,0,0,0,0,0,0,0,0
zenodo,0,0,27,375,1275,21,371,0,4,0,0,0,449,0


In [ ]:

discarded_by_source = (
    discarded_df
    .groupby('src_repo')
    .size()
    .reset_index(name='discarded_count')
    .sort_values('discarded_count', ascending=False)
    .reset_index(drop=True)
)

# Display the results
print("Count of discarded rows by source:")
print(discarded_by_source)

total_by_source = (
    pd.concat([iris_df, discarded_df])
    .groupby('src_repo')
    .size()
    .reset_index(name='total_count')
)

stats_comparison = discarded_by_source.merge(total_by_source, on='src_repo')
stats_comparison['discard_rate_%'] = (stats_comparison['discarded_count'] / stats_comparison['total_count'] * 100).round(2)

print("\nDetailed statistics by source:")
print(stats_comparison)

Count of discarded rows by source:
  src_repo  discarded_count
0     iris           407558
1   zenodo             2912
2  amsacta              657

Detailed statistics by source:
  src_repo  discarded_count  total_count  discard_rate_%
0     iris           407558       412267           98.86
1   zenodo             2912         5434           53.59
2  amsacta              657         1109           59.24
